<a href="https://colab.research.google.com/github/soy-dice/open_campus/blob/main/oc_phylogeny_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# オープンキャンパス

## 身近な植物の系統関係を調べる

固定の7種 + 10種類の植物から3種を選択して系統樹を推定

葉緑体ゲノムのマチュラーゼKという遺伝子のアラインメント済みのfastaをダウンロードする

系統樹推定はiqtree, 描画は基本Biopython + Matplotlib

In [ ]:
# @title 配列 ツールのDL(事前に実行しておく)
%%capture
# 配列のダウンロード
!wget -O oc_demo.fasta https://gist.githubusercontent.com/soy-dice/ce2dcccc786579413515a96267f5fc39/raw/906d723eaae2c7c6b828491b3b2fa3a9eab12916/gistfile1.txt
!wget -O species_photos_pack.zip "https://gist.github.com/soy-dice/1c4aced7223ff48b43f79cf241a08b2c/raw/species_photos_pack.zip"

# 必要なツールのインストール
!apt install iqtree # 系統樹推定用
!pip install -q toytree biopython japanize-matplotlib # 描画用

In [ ]:
# @title  系統樹に追加する植物を選んでください
# @markdown 比較したい植物を3つ選択してください。

from Bio import SeqIO

植物1 = "トマト" #@param ["サツマイモ", "ニンジン", "トマト", "ヒマワリ", "タバコ", "メロン", "ダイズ", "シソ", "オクラ", "ゴボウ"]
植物2 = "ヒマワリ" #@param ["サツマイモ", "ニンジン", "トマト", "ヒマワリ", "タバコ", "メロン", "ダイズ", "シソ", "オクラ", "ゴボウ"]
植物3 = "シソ" #@param ["サツマイモ", "ニンジン", "トマト", "ヒマワリ", "タバコ", "メロン", "ダイズ", "シソ", "オクラ", "ゴボウ"]

# 変数に代入し直す
guest_1 = 植物1
guest_2 = 植物2
guest_3 = 植物3

print(f"選択された種: 【{guest_1}】【{guest_2}】【{guest_3}】")

# 日本語名から学名へ変換
name_to_id = {
    "サツマイモ": "Ipomoea_batatas",
    "ニンジン": "Daucus_carota",
    "トマト": "Solanum_lycopersicum",
    "ヒマワリ": "Helianthus_annuus",
    "タバコ": "Nicotiana_tabacum",
    "メロン": "Cucumis_melo",
    "ダイズ": "Glycine_max",
    "シソ": "Perilla_frutescens",
    "オクラ": "Abelmoschus_esculentus",
    "ゴボウ": "Arctium_lappa",
}

# コア7種
core_species = [
    "Persea_americana",      # アボカド
    "Solanum_tuberosum",     # ジャガイモ
    "Capsicum_annuum",       # ピーマン
    "Lactuca_sativa",        # レタス
    "Brassica_oleracea",     # キャベツ
    "Solanum_melongena",     # ナス
    "Raphanus_sativus",      # ダイコン
]

# 選ばれた日本語名を学名に変換して結合
selected_species = [name_to_id[guest_1], name_to_id[guest_2], name_to_id[guest_3]]
target_species = core_species + selected_species

# 学名⇔和名の変換辞書と、ヘッダー文字列から学名を探す関数
# (アラインメント表示・系統樹描画の両方で共通して使う)
name_map = {
    "Persea_americana": "アボカド", "Solanum_tuberosum": "ジャガイモ",
    "Capsicum_annuum": "ピーマン", "Lactuca_sativa": "レタス",
    "Brassica_oleracea": "キャベツ", "Solanum_melongena": "ナス",
    "Raphanus_sativus": "ダイコン",
}
for jp_name, sci_name in name_to_id.items():
    name_map[sci_name] = jp_name

def find_species(header):
    """ヘッダー文字列に含まれる学名を探す"""
    return next((sp for sp in name_map if sp in header), None)

# fastaから使用する種を選択
input_fasta = "oc_demo.fasta" # 事前に全種をまとめたファイル
output_fasta = "to_run.fasta" # IQ-TREEに投げる用のファイル


# 抽出条件の関数を作成
def filter_records(fasta_path, targets):
    for record in SeqIO.parse(fasta_path, "fasta"):
        if any(target in record.description for target in targets):
            yield record

SeqIO.write(filter_records(input_fasta, target_species), output_fasta, "fasta")

In [ ]:
# @title 配列の可視化
import matplotlib.pyplot as plt
import numpy as np
import japanize_matplotlib
from Bio import SeqIO

zoom_len = 60

# データ読み込み
# この時点で先頭60文字だけ切り出す
seqs_by_species = {find_species(rec.id): str(rec.seq)[:zoom_len] for rec in SeqIO.parse("to_run.fasta", "fasta")}
display_order = core_species + selected_species
labels = [name_map[sp] for sp in display_order]

# アミノ酸を数値（0〜7）に変換
aa_color_id = {c: i for i, group in enumerate(["GAST", "C", "DENQ", "KRH", "ILMV", "FYW", "P", "-"]) for c in group}
matrix = np.array([[aa_color_id.get(c.upper(), 7) for c in seqs_by_species[sp]] for sp in display_order])

# 3. 描画
fig, ax = plt.subplots(figsize=(16, 0.6 * len(display_order) + 1))
ax.imshow(matrix, aspect="auto", cmap="Set3")

# ラベルやタイトルをまとめて設定
ax.set_yticks(range(len(labels)), labels=labels, fontsize=12)
ax.set(xlabel=f"配列の位置(1〜{zoom_len}残基目)", title="配列アラインメント(拡大)")

# 文字の重ね書き
for i, sp in enumerate(display_order):
    for j, c in enumerate(seqs_by_species[sp]):
        ax.text(j, i, c, ha="center", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("alignment_zoom.png", dpi=150)
plt.show()

In [ ]:
# @title 系統樹の推定
!iqtree2 -s to_run.fasta -m LG -pre result -redo

In [ ]:
# @title 系統樹の可視化
from Bio import Phylo
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.image as mpimg
import japanize_matplotlib
import os, shutil

BRANCH_COLOR = "#9aa5a0"
NODE_COLOR = "#5c8374"
IMG_DIR = "species_photos"


def load_tree():
    # treeの読み込み、outgroupの指定
    # そのままだとうまくいかなかったので、手動で/2して半分の地点においている
    tree = Phylo.read("result.treefile", "newick")
    outgroup = next(c for c in tree.find_clades() if find_species(c.name or "") == "Persea_americana")
    tree.root_with_outgroup(outgroup, outgroup_branch_length=(outgroup.branch_length or 0.0) / 2)
    return tree


def prepare_photos():
    # zipを解凍
    if not os.path.exists(IMG_DIR) or not os.listdir(IMG_DIR):
        shutil.rmtree(IMG_DIR, ignore_errors=True)
        shutil.unpack_archive("species_photos_pack.zip", IMG_DIR)


def compute_coords(tree):
    # X座標: tree.depths()で根からの距離を取得
    raw_x = tree.depths()
    scale = 6.0 / max(raw_x.values())
    x_coords = {c: x * scale for c, x in raw_x.items()}

    # Y座標: 末端ノードを下から順に並べ、親は子の中間地点にする
    leaves = tree.get_terminals()
    y_coords = {leaf: i for i, leaf in enumerate(leaves)}

    def set_y(clade):
        if not clade.is_terminal():
            y_coords[clade] = sum(set_y(c) for c in clade.clades) / len(clade.clades)
        return y_coords[clade]
    set_y(tree.root)

    return x_coords, y_coords, leaves

def draw_branches(ax, clade, x_coords, y_coords):
    x, y = x_coords[clade], y_coords[clade]
    for child in clade.clades:
        cx, cy = x_coords[child], y_coords[child]
        ax.plot([x, x], [y, cy], color=BRANCH_COLOR, lw=2)
        ax.plot([x, cx], [cy, cy], color=BRANCH_COLOR, lw=2)
        draw_branches(ax, child, x_coords, y_coords)
    if clade.clades:
        ax.scatter([x], [y], s=28, color=NODE_COLOR, edgecolors="white", zorder=3)


def draw_leaf_labels(ax, leaves, x_coords, y_coords, label_x):
    for leaf in leaves:
        x, y = x_coords[leaf], y_coords[leaf]
        sp = find_species(leaf.name or "")
        label = name_map.get(sp, leaf.name)
        color = "#e74c3c" if sp in selected_species else "#2c3e50"

        # 枝とラベルを繋ぐ点線
        ax.plot([x, label_x - 0.1], [y, y], ls=":", color="#bbb", lw=1)

        # 写真を読み込んで描画
        img = mpimg.imread(f"{IMG_DIR}/{sp}.png")
        ax.add_artist(AnnotationBbox(OffsetImage(img, zoom=0.35), (label_x, y), frameon=False))

        # テキスト
        ax.text(label_x + 0.55, y, label, fontsize=13, fontweight="bold", va="center", color=color)


def main():
    tree = load_tree()
    prepare_photos()  # 写真の解凍のみ実行
    x_coords, y_coords, leaves = compute_coords(tree)

    fig, ax = plt.subplots(figsize=(13, max(6, len(leaves) * 0.9)))
    draw_branches(ax, tree.root, x_coords, y_coords)

    max_x = max(x_coords[leaf] for leaf in leaves)
    label_x = max_x * 1.15 + 0.4

    draw_leaf_labels(ax, leaves, x_coords, y_coords, label_x)

    ax.set_xlim(-max_x * 0.08, label_x + 1.5)
    ax.set_ylim(-1.5, len(leaves) + 0.5)
    ax.invert_yaxis()
    ax.axis("off")
    plt.title("系統樹", fontsize=16, fontweight="bold", pad=15)
    plt.tight_layout()
    plt.savefig("phylogeny_with_photos.png", dpi=150)
    plt.show()

main()